In [1]:
!pip install cloudscraper
!pip install bs4
!pip install pandas

import cloudscraper
from bs4 import BeautifulSoup
import time
import random
import pandas as pd
import re


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
def realizar_scraping():
    scraper = cloudscraper.create_scraper(
        browser={
            'browser': 'chrome',
            'platform': 'windows',
            'desktop': True
        }
    )
    
    base_url = "https://backloggd.com"
    links_dos_jogos = []
    pagina_atual = 1

    print("--- ETAPA 1: Coletando links dos jogos ---")
    
    limite_jogos = 200
    
    while len(links_dos_jogos) < limite_jogos:
        if pagina_atual > 10:
            print("Passou de 10 páginas e não achou tudo. Parando por segurança.")
            break

        url_paginada = f"{base_url}/games/lib/popular/?page={pagina_atual}"
        print(f"Acessando página {pagina_atual} de populares...")
        
        response = scraper.get(url_paginada)
        
        if response.status_code != 200:
            print(f"Erro ao acessar! Status: {response.status_code}.")
            break
            
        soup = BeautifulSoup(response.text, "html.parser")
        
        links_html = soup.find_all("a", class_="cover-link")
        
        if not links_html:
            print("Nenhum link encontrado! O site pode ter bloqueado ou mudado o HTML.")
            break
            
        for link in links_html:
            caminho = link.get("href")
            if caminho and caminho not in links_dos_jogos:
                links_dos_jogos.append(caminho)
            
            if len(links_dos_jogos) == limite_jogos:
                break
                
        pagina_atual += 1
        time.sleep(random.uniform(2.5, 4.5))

    if links_dos_jogos:
        print(f"\nSucesso! {len(links_dos_jogos)} links coletados.")
        print("\n--- ETAPA 2: Extraindo dados de cada jogo ---")
        
        dados_finais = []

        for index, caminho in enumerate(links_dos_jogos, start=1):
            url_jogo = base_url + caminho
            print(f"[{index}/{limite_jogos}] Extraindo dados de: {url_jogo}")
            
            res = scraper.get(url_jogo)
            
            if res.status_code == 200:
                soup = BeautifulSoup(res.text, "html.parser")
                
                # 1. NOME
                nome_tag = soup.find("h1")
                nome = nome_tag.get_text(strip=True) if nome_tag else "N/A"
                
                # 2. DESCRIÇÃO
                descricao = "N/A"
                div_summary = soup.find("div", id="collapseSummary")
                if div_summary:
                    p_tag = div_summary.find("p", class_="mb-0")
                    # Pegamos o texto usando espaço como separador
                    desc_bruta = p_tag.get_text(separator=" ", strip=True) if p_tag else div_summary.get_text(separator=" ", strip=True)
                    # A mágica para o Excel: Troca enter/quebra de linha (\n) e múltiplos espaços por 1 espaço simples!
                    descricao = re.sub(r'\s+', ' ', desc_bruta)
                
                # 3. TAGS
                tags_html = soup.find_all("a", class_="game-details-value")
                tags_brutas = [tag.get_text(strip=True) for tag in tags_html if "genre" in tag.get("href", "")]
                tags = list(dict.fromkeys(tags_brutas)) # Remove duplicatas (PC vs Mobile)

                # 4. DATA DE LANÇAMENTO
                # Busca qualquer link que contenha 'release_year' no href
                data_tag = soup.find("a", href=re.compile(r"release_year"))
                data_lancamento = data_tag.get_text(strip=True) if data_tag else "N/A"

                # 5. DESENVOLVEDORAS / PUBLISHERS
                empresas_tags = soup.find_all("a", href=re.compile(r"/company/"))
                # Pega os nomes e remove as duplicatas (PC vs Mobile) mantendo a ordem
                empresas = list(dict.fromkeys([emp.get_text(strip=True) for emp in empresas_tags]))

                # 6. PLATAFORMAS
                plataformas_tags = soup.find_all("a", class_="game-page-platform")
                # Remove duplicatas
                plataformas = list(dict.fromkeys([p.get_text(strip=True) for p in plataformas_tags]))

                # 7. NOTA MÉDIA
                nota = "N/A"
                rating_div = soup.find("div", id="game-rating")
                if rating_div:
                    h1 = rating_div.find("h1")
                    if h1:
                        nota = h1.get_text(strip=True)

                # 8. TEMPOS DE JOGO
                tempos_brutos = []
                for tp in soup.find_all("div", class_="time-played"):
                    val = tp.find(class_="element-revealed")
                    lbl = tp.find(class_="label")
                    if val and lbl:
                        tempos_brutos.append(f"{lbl.get_text(strip=True)}: {val.get_text(strip=True)}")
                tempos_lista = list(dict.fromkeys(tempos_brutos))

                # 9. REVIEWS E LIKES
                qtd_reviews = "N/A"
                qtd_likes = "N/A"
                
                # Varre os blocos onde esses números costumam ficar e checa pelo texto (Reviews ou Likes)
                for container in soup.find_all("div", class_="center-container"):
                    p_tag = container.find("p")
                    h3_tag = container.find("h3")
                    if p_tag and h3_tag:
                        texto_p = p_tag.get_text(strip=True).lower()
                        if "reviews" in texto_p:
                            qtd_reviews = h3_tag.get_text(strip=True)
                        elif "likes" in texto_p:
                            qtd_likes = h3_tag.get_text(strip=True)

                # SALVANDO OS DADOS
                dados_finais.append({
                    "Posicao": index,
                    "Nome": nome,
                    "Lançamento": data_lancamento,
                    "Desenvolvedora/Publisher": ", ".join(empresas) if empresas else "N/A",
                    "Plataformas": ", ".join(plataformas) if plataformas else "N/A",
                    "Nota Média": nota,
                    "Tempo de Jogo": " | ".join(tempos_lista) if tempos_lista else "N/A",
                    "Qtd Reviews": qtd_reviews,
                    "Qtd Likes": qtd_likes,
                    "Tags": ", ".join(tags) if tags else "N/A",
                    "Descricao": descricao
                })
                
            else:
                print(f"Falha ao carregar o jogo. Erro {res.status_code}")
                
            time.sleep(random.uniform(3.0, 5.0)) # Tempo seguro para evitar block

        if dados_finais:
            print("\n--- RESULTADO FINAL ---")
            df = pd.DataFrame(dados_finais)
            print(df)
            df.to_csv("top_jogos_backloggd.csv", index=False, encoding='utf-8-sig') # utf-8-sig evita erro com caracteres no Excel

realizar_scraping()

--- ETAPA 1: Coletando links dos jogos ---
Acessando página 1 de populares...
Acessando página 2 de populares...
Acessando página 3 de populares...
Acessando página 4 de populares...

Sucesso! 200 links coletados.

--- ETAPA 2: Extraindo dados de cada jogo ---
[1/200] Extraindo dados de: https://backloggd.com/games/grand-theft-auto-v/
[2/200] Extraindo dados de: https://backloggd.com/games/red-dead-redemption-2/
[3/200] Extraindo dados de: https://backloggd.com/games/elden-ring/
[4/200] Extraindo dados de: https://backloggd.com/games/minecraft-java-edition/
[5/200] Extraindo dados de: https://backloggd.com/games/the-legend-of-zelda-breath-of-the-wild/
[6/200] Extraindo dados de: https://backloggd.com/games/hollow-knight/
[7/200] Extraindo dados de: https://backloggd.com/games/god-of-war--1/
[8/200] Extraindo dados de: https://backloggd.com/games/portal-2/
[9/200] Extraindo dados de: https://backloggd.com/games/portal/
[10/200] Extraindo dados de: https://backloggd.com/games/cyberpunk-2